# 04 · Phenology metrics per year and topographic zone

Loads the cached Sentinel-2 statistics, filters dates by quality and derives the start, peak and end of the autumn yellowing season for every year and zone with two methods, the Whittaker smoother and the double logistic. It compares the two, projects the chosen dates a few years ahead and relates them to elevation and aspect.

| Section | Content |
|---|---|
| 1 | load and filter the cached statistics |
| 2 | the indices over the years |
| 3 | Whittaker and double-logistic curves per year and zone |
| 4 / 5 / 6 | start / peak / end of the season: both methods |
| 7 | comparison of the two methods and the chosen date per date type; 7c: check at a local site |
| 8 | OLS trend and projection of the chosen dates |
| 9 | chosen dates vs elevation and aspect |
| 10 | STL decomposition of the indices |
| 11 | export |

- **Settings:** `config.py` (years, date filters, fit settings, `SEASON_METHODS` and `SEASON_CHOICE`)
- **Reads:**
  - `cache/<AOI>_zones/` (from `01_download_sentinel2`)
  - `cache/validation/<AOI>/` (from `02_download_validation`, optional)
  - `cache/<AOI>_sites/` (from `01_download_sentinel2` section 6, optional)
- **Writes** to `results/<AOI>/`:
  - `season_dates.csv`: start / mid / end of both methods per zone and year
  - `season_method_comparison.csv`: the comparison of section 7
  - `season_selected.csv`: the chosen start / peak / end per zone and year (read by `05_climate_analysis`)
  - `season_projection.csv`: the projection of section 8
  - `site_dates.csv`, `site_check.csv`: dates of the areas around local sites and their comparison with the sites' observations (7c)
  - `zones.csv`, `topography_effects.csv`
  - `ndvi_timeseries_all_dates.csv`, `ndvi_timeseries_clean.csv`
  - `report_figures/`: the figures recommended for the report (numbered in report order; with the maps of `02` and the figures of `05`)

Years whose autumn is not fully cached yet (`SEASON_COMPLETE_DOY`, e.g. the current year) are left out of all plots, tables, trends and models.

Runs offline in about a minute. Sources are listed at the end.

## Yellowing season: start, peak and end

The aim is three dates per year and zone, each from two methods (`SEASON_METHODS` in `config.py`):

| Date | Meaning | `Whittaker` | `double logistic` | Checked against (section 7) |
|---|---|---|---|---|
| **Start** | needles begin to lose chlorophyll | GRVI, 90 % of the autumn amplitude left | GRVI, 90 % left | order |
| **Peak** | canopy most yellow: chlorophyll gone, needles still on | GRVI, 50 % left | NDVI, 50 % left | needle colouring 50 % (MeteoSwiss, local sites) |
| **End** | needles have fallen | GRVI, 15 % left | NDVI, 15 % left | needle drop 50 % (MeteoSwiss, local sites) |

- **Methods:** the Whittaker smoother follows the observations without assuming a curve shape. The double logistic, the function TIMESAT fits for HR-VPP, assumes the shape of the seasonal curve and gives an independent, parametric second estimate.
- **Index per method:** the Whittaker dates use GRVI. The double logistic uses GRVI for the start; for the peak and the end its GRVI dates vary strongly between years, so it uses NDVI there. PSRI (Merzlyak et al. 1999) and RCC are shown in section 2 but give no dates: the PSRI dates on the multidate AOI come later at higher elevation, opposite to the expected gradient.
- **Signals:** greenness (GRVI, NDVI) falls as chlorophyll breaks down; the change from green to yellow shows mainly in the visible bands (green falls relative to red; Motohka et al. 2010). Yellow needles still reflect NIR strongly, so NDVI drops mostly with needle fall.
- **Thresholds:** 90 / 50 / 15 % of the amplitude follow MODIS MCD12Q2 (senescence / greendown midpoint / dormancy; Gray et al. 2019) and HR-VPP (end of season at 15 %; Tian et al. 2021).
- **Expected:** start < peak < end in every year and zone; earlier at higher elevation.
- **Limits:** after mid-October there are few clear observations (low sun turns north slopes SCL "dark", clouds, early snow), so many dates, the end above all, lie in a data gap where the curve only interpolates.

In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings

from matplotlib.colors import LightSource
from PIL import Image
from scipy import sparse
from scipy.optimize import OptimizeWarning, curve_fit
from scipy.sparse.linalg import spsolve
from statsmodels.tsa.seasonal import STL

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import (clean_frame, date_title, incomplete_years, index_frame, load_stats, load_zone_table,
                             save_report_figure)

INDEX_COLORS = {"NDVI": "darkgreen", "GRVI": "darkorange", "PSRI": "purple", "RCC": "firebrick"}
INDEX_MARKERS = {"NDVI": "o", "GRVI": "s", "PSRI": "^", "RCC": "v"}
ASPECT_CODES = {"N": 0, "S": 1, "flat": 2}   # aspect classes as stored in zones.tif (01)
ASPECT_COLORS = {"N": "tab:blue", "S": "tab:orange", "flat": "grey"}
ASPECT_NAMES = {"N": "north-facing", "S": "south-facing", "flat": "flat"}   # in figure labels
METHOD_COLORS = {"Whittaker": "tab:green", "double logistic": "tab:orange"}
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | results: {OUT_DIR}")

## 1. Load and filter the cached statistics

A date is kept for a zone if at least `MIN_VALID_FRACTION` of the zone's pixels are clear (and it passes `MAX_SCENE_CLOUD` / `MAX_SCL_FRACTION`). With `VALID_SHARE_OF = "covered"`, this and the other SCL shares count only the zone pixels inside the scene's footprint: every second scene comes from the edge of a neighbouring orbit's swath and covers only part of the AOI, and such a scene passes if its covered part is clear and holds at least `MIN_COVERED_PIXELS` pixels. `df`, `df_clean` and so on refer to the whole AOI (zone `"all"`). `clean_by_zone[index][zone]` holds the cleaned median series of every zone. `INCOMPLETE` lists the years whose cache ends before `SEASON_COMPLETE_DOY`. Indices of `INDICES` that the cache lacks are skipped.

In [ ]:
stats_all = load_stats()
zone_table = load_zone_table()
ZONES = list(zone_table.index)   # "all" first, then elevation band x aspect
INCOMPLETE = incomplete_years(stats_all)
if INCOMPLETE:
    print(f"Autumn not fully cached, left out: {INCOMPLETE}")
    stats_all = stats_all[~stats_all.index.year.isin(INCOMPLETE)]

missing = [n for n in INDICES if f"{n.lower()}_median" not in stats_all or stats_all[f"{n.lower()}_median"].isna().all()]
if missing:
    print(f"Not in the cache, skipped: {missing} (re-run 01_download_sentinel2 to add them)")
INDICES = [n for n in INDICES if n not in missing]

frames = {name: index_frame(stats_all, name.lower()) for name in INDICES}
clean = {name: clean_frame(stats_all, name.lower()) for name in INDICES}
df, df_clean = frames["NDVI"], clean["NDVI"]
for name in INDICES:
    print(f"{name}: {len(clean[name])} of {len(frames[name])} dates kept")

clean_by_zone = {name: {zone: clean_frame(stats_all, name.lower(), zone)[f"{name.lower()}_median"] for zone in ZONES}
                 for name in INDICES}
display(zone_table[["n_pixels", "elev_mean", "slope_mean", "northness_mean"]].round(2).join(
    pd.DataFrame({name: {z: len(s) for z, s in by_zone.items()} for name, by_zone in clean_by_zone.items()})
      .add_prefix("dates_")))

### 1b. Check: partial-swath scenes vs full scenes

A partial scene always sees the same strip of a zone, from a different viewing angle. If that strip is not representative, its zone median is offset from the full-scene medians. Each kept partial scene (covering < 90 % of the zone) is paired with the nearest kept full scene at most `PAIR_MAX_DAYS` days away (usually 2), both without snow.

- **`bias`:** mean difference partial − full; near 0 = no systematic offset
- **`mad`:** median absolute difference, including real change over the days in between
- **`full_mad`:** the same for pairs of full scenes ≤ 5 days apart, as the noise to compare against
- **`bias_share`:** `bias` relative to the zone's seasonal range (5th–95th percentile)

A `bias_share` of a few percent shifts a transition date by much less than a day. If it is large for a zone, set `VALID_SHARE_OF = "zone"` or raise `MIN_COVERED_PIXELS`.

In [ ]:
PAIR_MAX_DAYS = 3
PARTIAL_COVER = 0.9   # scenes covering less of the zone count as partial


def paired_diff(a, b, max_days):
    # a - b for each date of a and the nearest other date of b at most max_days away (series indexed by date)
    m = pd.merge_asof(a.rename("a").to_frame(), b.rename("b").to_frame(), left_index=True, right_index=True,
                      direction="nearest", tolerance=pd.Timedelta(days=max_days), allow_exact_matches=False)
    return (m.a - m.b).dropna()


rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)   # pandas' merge_asof tolerance on recent numpy
    for name in INDICES:
        col = f"{name.lower()}_median"
        for zone in ZONES:
            f = clean_frame(stats_all, name.lower(), zone)
            f = f[f.scl_snow_fraction <= MAX_SNOW_FRACTION]
            partial = f.covered_fraction < PARTIAL_COVER
            diff = paired_diff(f.loc[partial, col], f.loc[~partial, col], PAIR_MAX_DAYS)
            full_diff = paired_diff(f.loc[~partial, col], f.loc[~partial, col], 5)
            spread = f[col].quantile(0.95) - f[col].quantile(0.05)
            rows.append({"index": name, "zone": zone, "partial_kept": int(partial.sum()), "n_pairs": len(diff),
                         "bias": diff.mean(), "mad": diff.abs().median(), "full_mad": full_diff.abs().median(),
                         "bias_share": diff.mean() / spread})
partial_check = pd.DataFrame(rows).set_index(["index", "zone"])
print(f"VALID_SHARE_OF = {VALID_SHARE_OF!r}, MIN_COVERED_PIXELS = {MIN_COVERED_PIXELS}")
display(partial_check.round(3))

## 2. The indices over the years (whole AOI)

- **Top:** every index over the whole period: all dates (grey), the dates that pass the filter (colour) and a centred moving average (`SMOOTH_WINDOW` × 5 days) on a regular 5-day grid, with gaps of up to 30 days interpolated. Shaded: preliminary years.
- **Bottom:** the moving average of each year over the snow-free season (`SEASON_WINDOW`), one line per year.

The moving average is only for display; the date methods work on the observations directly.

In [ ]:
def smooth(series):
    # Centred moving average on a regular 5-day grid; gaps of up to 30 days interpolated, longer ones left empty
    regular = series.resample("5D").mean().interpolate(method="time", limit=6)
    rolled = regular.rolling(SMOOTH_WINDOW, center=True, min_periods=3).mean()
    return rolled.where(regular.notna())


def clear_limits(name, margin=0.15):
    # y limits from the clear dates (snow and cloud values of the unfiltered dates would squash the plot)
    lo, hi = medians[name].quantile([0.005, 0.995])
    return lo - margin * (hi - lo), hi + margin * (hi - lo)


medians = {name: clean[name][f"{name.lower()}_median"] for name in INDICES}
smoothed = {name: smooth(s) for name, s in medians.items()}

fig, axes = plt.subplots(len(INDICES), 1, figsize=(13, 2.4 * len(INDICES)), sharex=True, squeeze=False)
for ax, name in zip(axes[:, 0], INDICES):
    ax.scatter(frames[name].index, frames[name][f"{name.lower()}_median"], s=5, color="lightgrey", label="all dates")
    ax.scatter(medians[name].index, medians[name], s=8, color=INDEX_COLORS[name], label="clear dates")
    ax.plot(smoothed[name].index, smoothed[name], color="black", lw=1.2,
            label=f"{SMOOTH_WINDOW * 5}-day moving average")
    ax.set_ylim(*clear_limits(name))
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
axes[0, 0].legend(loc="lower left", ncol=3, fontsize=8)
fig.suptitle("Vegetation indices over the years (whole larch area)")
plt.tight_layout()
plt.show()

years = sorted(set(stats_all.index.year))
year_colors = dict(zip(years, plt.cm.viridis(np.linspace(0, 1, len(years)))))
fig, axes = plt.subplots(1, len(INDICES), figsize=(4.5 * len(INDICES), 3.8), squeeze=False)
for ax, name in zip(axes[0], INDICES):
    s = smoothed[name]
    for year, color in year_colors.items():
        y = s[s.index.year == year]
        ax.plot(y.index.dayofyear, y, color=color, lw=1.2, label=str(year))
    ax.set_xlim(*SEASON_WINDOW)
    ax.set_ylim(*clear_limits(name))
    ax.set_title(name)
    ax.set_xlabel("day of year")
    ax.grid(alpha=0.3)
axes[0, -1].legend(fontsize=7, ncol=2)
fig.suptitle(f"Vegetation indices per year (whole larch area, {SMOOTH_WINDOW * 5}-day average)")
plt.tight_layout()
save_report_figure(fig, '06_indices_seasonal_course')
plt.show()

## 3. Curves fitted per year and zone

Both date methods read their dates off a curve fitted to the clear observations of each index, zone and year:

| Curve | Fit | Source |
|---|---|---|
| `Whittaker` | weighted Whittaker smoother, daily, over the whole series (3b) | Eilers 2003; Atzberger & Eilers 2011 |
| `double logistic` | whole snow-free season, base fixed, weighted (3c) | Beck et al. 2006; TIMESAT / HR-VPP (Jönsson & Eklundh 2004) |

Curves are fitted for the indices of `SEASON_METHODS` only. 3b and 3c each end with a figure of their curves for the whole AOI: one row per index, one column per year, with the clear dates (× = weight 0: snow or outlier), the post-season level (dotted) and the method's start / mid / end (vertical lines). The candidates' dates go into one table, `season_dates` (3d). Overview of the methods: Zeng et al. 2020.

### 3a. Observations, weights, outliers and the post-season level

- **Weights** follow the HR-VPP quality classes: 1 for a good date (clear share ≥ `GOOD_VALID_FRACTION`), 0.5 for a usable one, 0 for a date with snow on more than `MAX_SNOW_FRACTION` of the zone.
- **Outliers** follow the HR-VPP rule: a date is dropped (weight 0) if its residual from a stiff smoothing curve exceeds `OUTLIER_FACTOR` × the global and × the local (91-day) median residual.
- **Post-season level ("base"):** the snow-free level after needle fall, the median of the clear dates in `POST_WINDOW` over all years. It is held fixed in the fits instead of being fitted per year, as Beck et al. (2006) do with the winter NDVI. Without a fixed base, a year with no data after the decline leaves the lower level unconstrained. Zones with fewer than `MIN_BASE_OBS` such dates use the whole AOI's level.

In [ ]:
# Candidates of SEASON_METHODS as (date type, method, index, metric); curves are fitted for their indices only
METHODS = ["Whittaker", "double logistic"]
CANDIDATES = [(target, *(part.strip() for part in c.split("/"))) for target, cs in SEASON_METHODS.items() for c in cs]
for c in CANDIDATES:
    if c[1] not in METHODS or c[3] not in AMPLITUDE_FRACTIONS:
        raise ValueError(f"SEASON_METHODS[{c[0]!r}]: {' / '.join(c[1:])!r} needs a method of {METHODS} "
                         f"and a metric of {list(AMPLITUDE_FRACTIONS)}")
CURVE_INDICES = [n for n in INDICES if n in {c[2] for c in CANDIDATES}]
missing = sorted({c[2] for c in CANDIDATES} - set(CURVE_INDICES))
if missing:
    print(f"Indices of SEASON_METHODS not in the cache, their candidates are skipped: {missing}")


def obs_frame(name, zone, stats=None, zones=None):
    # Clear observations of one index and zone: y (zone median), w (weight), year, doy. Weights as the HR-VPP
    # quality classes (ATBD; Tian et al. 2021): 1 = good, 0.5 = usable, 0 = snow (kept for the plots, not fitted).
    # stats / zones: statistics and zone table of another cache (the site areas of 7c); default: the zones
    f = clean_frame(stats_all if stats is None else stats, name.lower(), zone, zones)
    obs = pd.DataFrame({"y": f[f"{name.lower()}_median"],
                        "w": np.where(f.valid_fraction >= GOOD_VALID_FRACTION, 1.0, 0.5)}, index=f.index)
    obs.loc[f.scl_snow_fraction > MAX_SNOW_FRACTION, "w"] = 0.0
    obs = obs.dropna(subset=["y"])
    return obs.assign(year=obs.index.year, doy=obs.index.dayofyear, outlier=False)


def base_levels(name):
    # Post-season level per zone, held fixed in all fits (Beck et al. 2006): median of the weighted dates in
    # POST_WINDOW over all years. Fallbacks: the whole AOI's level; then, as HR-VPP sets its base level, the 5th
    # percentile of all clear dates (95th for rising indices, whose post-season level is the high one)
    rows = {}
    for zone in ZONES:
        o = obs_by_zone[name][zone]
        post = o[o.doy.between(*POST_WINDOW) & (o.w > 0)]
        rows[zone] = {"level": post.y.median(), "n_dates": len(post), "source": "zone"}
    for zone, r in rows.items():
        if r["n_dates"] < MIN_BASE_OBS and rows["all"]["n_dates"] >= MIN_BASE_OBS:
            r.update(level=rows["all"]["level"], source="whole AOI")
        elif r["n_dates"] < MIN_BASE_OBS:
            q = 0.05 if EXPECT_DECREASE[name] else 0.95
            r.update(level=obs_by_zone[name][zone].y.quantile(q), source=f"{q:.0%} percentile")
    return pd.DataFrame(rows).T.astype({"level": float, "n_dates": int})


# ---- Weighted Whittaker smoother (also used for the outlier test, so it is defined here)
def whittaker(days, y, w, lam, n):
    # Eilers (2003), for vegetation indices Atzberger & Eilers (2011): on a daily grid of n days, minimise
    # sum w (y - z)^2 + lam * sum (2nd difference of z)^2. Days without an observation have weight 0, so irregular
    # dates and gaps need no interpolation beforehand; across a gap the curve becomes nearly a straight line.
    W, Y = np.zeros(n), np.zeros(n)
    np.add.at(W, days, w)
    np.add.at(Y, days, w * y)
    D = sparse.diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n - 2, n))
    return spsolve((sparse.diags(W) + lam * (D.T @ D)).tocsc(), Y)


def smooth_series(obs, lam, base):
    # Daily Whittaker curve (pd.Series over whole calendar years) through the weighted dates of obs. Long
    # off-season gaps are filled as in HR-VPP: if more than 90 days pass without a good date between the autumn
    # and the spring equinox, control points at the base level (weight 0.5) are added every 15 days.
    o = obs[obs.w > 0]
    start = pd.Timestamp(obs.index.min().year, 1, 1)
    end = pd.Timestamp(obs.index.max().year, 12, 31)
    grid = pd.date_range(start, end)
    days, y, w = (o.index - start).days.values, o.y.values, o.w.values
    control = [d for a, b in zip(days[:-1], days[1:]) if b - a > 90 for d in range(a + 15, b - 7, 15)
               if not 80 < grid[d].dayofyear < 266]
    days = np.concatenate([days, control]).astype(int)
    y = np.concatenate([y, np.full(len(control), base)])
    w = np.concatenate([w, np.full(len(control), 0.5)])
    return pd.Series(whittaker(days, y, w, lam, len(grid)), index=grid)


def flag_outliers(obs, base):
    # HR-VPP outlier rule (ATBD 4.2.1): residuals from a stiff curve; outlier if the residual exceeds
    # OUTLIER_FACTOR x the global median residual and OUTLIER_FACTOR x the median of the surrounding 91 days
    o = obs[obs.w > 0]
    curve = smooth_series(obs, max(WHITTAKER_LAMBDAS), base)
    res = (o.y - curve.reindex(o.index).values).abs()
    local = res.rolling("91D", center=True, min_periods=1).median()
    out = res.index[(res > OUTLIER_FACTOR * res.median()) & (res > OUTLIER_FACTOR * local)]
    obs = obs.copy()
    obs.loc[out, ["w", "outlier"]] = [0.0, True]
    return obs


# ---- Double logistic (3c) and the curve figure (3b, 3c)
def double_logistic(t, m, S, mS, A, mA, b):
    # Beck et al. (2006): b = off-season level (fixed), m = summer level, S / A = spring / autumn inflection day,
    # mS / mA = their rates (1/day)
    return b + (m - b) * (1 / (1 + np.exp(-mS * (t - S))) + 1 / (1 + np.exp(mA * (t - A))) - 1)


def fraction_day(t0, s, f):
    # Day on a logistic when a share f of the amplitude is left
    return t0 + s * np.log((1 - f) / f)


def fit_double_logistic(doy, y, w, b):
    summer = y[(doy >= SUMMER_WINDOW[0]) & (doy <= SUMMER_WINDOW[1])]
    p0 = [np.clip(np.median(summer if len(summer) else y), -1.9, 1.9), 160, 0.1, 280, 0.1]
    bounds = ([-2, SEASON_WINDOW[0], 1 / 40, SUMMER_WINDOW[0], 1 / 40], [2, SUMMER_WINDOW[0], 1, SEASON_WINDOW[1], 1])
    popt, _ = curve_fit(lambda t, m, S, mS, A, mA: double_logistic(t, m, S, mS, A, mA, b), doy, y, p0=p0,
                        sigma=1 / np.sqrt(w), bounds=bounds, maxfev=20000)
    return popt


CURVE_DOY = np.arange(SEASON_WINDOW[0], SEASON_WINDOW[1] + 1)
DATE_STYLES = {"start": "--", "mid": "-", "end": "-."}
DATE_LABELS = {m: f"{ {'mid': 'peak'}.get(m, m)} ({AMPLITUDE_FRACTIONS[m] * 100:.0f} % of the autumn change left)"
               for m in DATE_STYLES}


def plot_curves(method, curve, rows, report_name=None):
    # Figure of 3b / 3c: curve(name, year) -> the whole-AOI curve over DOY (or None), rows -> the method's dates
    dates = {(r["index"], r["year"], r["metric"]): r["doy"] for r in rows if r["zone"] == "all"}
    fig, axes = plt.subplots(len(CURVE_INDICES), len(years), figsize=(2.3 * len(years), 2.3 * len(CURVE_INDICES) + 0.8),
                             sharex=True, sharey="row", squeeze=False)
    for i, name in enumerate(CURVE_INDICES):
        o_all = obs_by_zone[name]["all"]
        for j, year in enumerate(years):
            ax = axes[i, j]
            o = o_all[o_all.year == year]
            ax.scatter(o[o.w > 0].doy, o[o.w > 0].y, s=6, color=INDEX_COLORS[name])
            ax.scatter(o[o.w == 0].doy, o[o.w == 0].y, s=12, marker="x", color=INDEX_COLORS[name])
            z = curve(name, year)
            if z is not None:
                ax.plot(z.index, z.values, color=METHOD_COLORS[method], lw=1.2)
            for metric, ls in DATE_STYLES.items():
                if (name, year, metric) in dates:
                    ax.axvline(dates[(name, year, metric)], color="black", ls=ls, lw=0.8)
            ax.axhline(base[name].level["all"], color="grey", ls=":", lw=0.8)
            ax.grid(alpha=0.3)
            if i == 0:
                ax.set_title(str(year), fontsize=10)
            if i == len(CURVE_INDICES) - 1:
                ax.set_xlabel("day of year")
        axes[i, 0].set_ylabel(name)
        axes[i, 0].set_ylim(*clear_limits(name))
    axes[0, 0].set_xlim(*SEASON_WINDOW)
    handles = [plt.Line2D([], [], color=METHOD_COLORS[method], lw=1.5, label="fitted curve"),
               plt.Line2D([], [], ls="", marker="o", ms=4, color="grey", label="clear date"),
               plt.Line2D([], [], ls="", marker="x", ms=5, color="grey", label="left out (snow or outlier)"),
               plt.Line2D([], [], ls=":", lw=0.8, color="grey", label="post-season level")]
    handles += [plt.Line2D([], [], color="black", ls=ls, lw=0.8, label=DATE_LABELS[metric]) for metric, ls in DATE_STYLES.items()]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), fontsize=8)
    fig.suptitle(f"{method.capitalize()} curves per year (whole larch area)", fontsize=13)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    if report_name:
        save_report_figure(fig, report_name)
    plt.show()


obs_by_zone = {name: {zone: obs_frame(name, zone) for zone in ZONES} for name in INDICES}
base = {name: base_levels(name) for name in INDICES}
for name in INDICES:
    for zone in ZONES:
        obs_by_zone[name][zone] = flag_outliers(obs_by_zone[name][zone], base[name].level[zone])

display(pd.concat({name: b for name, b in base.items()}, axis=1).round(3))
print("Dates per index and zone: weight 1 / 0.5 / 0 (snow) / outliers")
display(pd.DataFrame({name: {zone: f"{(o.w == 1).sum()} / {(o.w == 0.5).sum()} / {((o.w == 0) & ~o.outlier).sum()} / "
                                   f"{o.outlier.sum()}" for zone, o in by_zone.items()}
                      for name, by_zone in obs_by_zone.items()}))

### 3b. Whittaker smoother and its dates

The weighted Whittaker smoother (Eilers 2003) fits a smooth daily curve to the irregular observations; Atzberger & Eilers (2011) introduced it for vegetation-index time series. It needs no gap interpolation beforehand, and one parameter, λ, sets the smoothness. λ is chosen per index from `WHITTAKER_LAMBDAS` by 10-fold cross-validation on the whole AOI (weighted RMSE of left-out dates), then used for every zone. HR-VPP assumes clear Sentinel-2 dates are unbiased and applies no upper-envelope correction; neither does this notebook.

**Dates.** Per year, the summer level `hi` is the curve's median in `SUMMER_WINDOW`; the base is the post-season level (3a). With `r` = share of the amplitude left:
- mid = first day after the summer window with `r ≤ 0.5`
- start = last day before mid with `r ≥ 0.9` (MCD12Q2: "last date above 90 %")
- end = first day after mid with `r ≤ 0.15`

A date that is not reached within the year stays empty.

In [ ]:
def choose_lambda(obs, base_level, k=10, seed=0):
    # Weighted RMSE of left-out good dates for every lam (k-fold cross-validation)
    good = obs[obs.w > 0]
    fold = np.random.default_rng(seed).integers(0, k, len(good))
    rmse = {}
    for lam in WHITTAKER_LAMBDAS:
        sq = []
        for i in range(k):
            test = good[fold == i]
            curve = smooth_series(obs.drop(index=test.index), lam, base_level)
            sq.append(test.w * (test.y - curve.reindex(test.index).values) ** 2)
        rmse[lam] = np.sqrt(pd.concat(sq).sum() / good.w.sum())
    return pd.Series(rmse)


cv_lambda = pd.DataFrame({name: choose_lambda(obs_by_zone[name]["all"], base[name].level["all"]) for name in CURVE_INDICES})
LAMBDA = cv_lambda.idxmin()
print("Cross-validated RMSE per lambda (whole AOI); chosen:", LAMBDA.to_dict())
display(cv_lambda.rename_axis("lambda").round(4))
wcurves = {name: {zone: smooth_series(obs_by_zone[name][zone], LAMBDA[name], base[name].level[zone]) for zone in ZONES}
           for name in CURVE_INDICES}

In [ ]:
def whittaker_dates(curve, hi, lo):
    # Start / mid / end on one year of a daily curve (index = DOY)
    r = (curve - lo) / (hi - lo)
    below_mid = r[(r.index > SUMMER_WINDOW[1]) & (r <= AMPLITUDE_FRACTIONS["mid"])]
    if below_mid.empty:
        return None
    mid = below_mid.index[0]

    def last_above(f):
        above = r[(r.index < mid) & (r >= f)]
        return above.index[-1] if not above.empty else np.nan

    def first_below(f):
        below = r[(r.index > mid) & (r <= f)]
        return below.index[0] if not below.empty else np.nan

    return {"start": last_above(AMPLITUDE_FRACTIONS["start"]), "mid": mid, "end": first_below(AMPLITUDE_FRACTIONS["end"])}


def zone_whittaker_dates(name, zone, obs, lo, curve_all):
    # Whittaker dates of one index and zone per year (rows of season_dates); curve_all = the zone's daily curve
    rows = []
    for year in sorted(obs.year.unique()):
        curve = curve_all[curve_all.index.year == year]
        curve.index = curve.index.dayofyear
        doy = obs[(obs.year == year) & (obs.w > 0)].doy.values
        hi = curve.loc[SUMMER_WINDOW[0]:SUMMER_WINDOW[1]].median()
        days = whittaker_dates(curve, hi, lo) if (hi > lo) == EXPECT_DECREASE[name] else None
        if days:
            rows += [{"index": name, "zone": zone, "year": year, "method": "Whittaker", "metric": m,
                      "doy": d, "n_obs": len(doy)} for m, d in days.items()]
    return rows


whittaker_rows = [row for name in CURVE_INDICES for zone in ZONES
                  for row in zone_whittaker_dates(name, zone, obs_by_zone[name][zone], base[name].level[zone],
                                                  wcurves[name][zone])]
print(f"{len(whittaker_rows)} Whittaker dates")


def whittaker_curve(name, year):
    z = wcurves[name]["all"]
    z = z[z.index.year == year]
    return pd.Series(z.values, index=z.index.dayofyear)


plot_curves("Whittaker", whittaker_curve, whittaker_rows, "07_whittaker_curves")

### 3c. Double logistic and its dates

- **Fit:** the function of Beck et al. (2006) over the whole snow-free season (`SEASON_WINDOW`), `y = b + (m − b)·(1/(1 + exp(−mS(t − S))) + 1/(1 + exp(mA(t − A))) − 1)`, with the base `b` fixed at the post-season level (3a) and the dates weighted. It is the function type TIMESAT fits for HR-VPP; only its autumn half (`A`, `mA`) gives dates here. A year needs at least `MIN_OBS_PER_YEAR` weighted dates between `AUTUMN_START_DOY` and the end of `POST_WINDOW`.
- **Dates:** a share `f` of the amplitude is left at `t = A + ln((1 − f) / f) / mA`; the 50 % point is `A`.

In [ ]:
def zone_dl_dates(name, zone, obs, lo):
    # Double-logistic fits of one index and zone per year: rows of season_dates, {(index, zone, year): parameters}
    rows, fits = [], {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", OptimizeWarning)
        for year, o in obs[obs.w > 0].groupby("year"):
            autumn = o[o.doy.between(AUTUMN_START_DOY, POST_WINDOW[1])]
            if len(autumn) < MIN_OBS_PER_YEAR:
                continue
            season = o[o.doy.between(*SEASON_WINDOW)]
            try:
                m, S, mS, A, mA = fit_double_logistic(season.doy.values.astype(float), season.y.values,
                                                      season.w.values, lo)
            except RuntimeError:
                continue
            fits[(name, zone, year)] = (m, S, mS, A, mA, lo)
            rows += [{"index": name, "zone": zone, "year": year, "method": "double logistic", "metric": metric,
                      "doy": fraction_day(A, 1 / mA, f), "n_obs": len(season)}
                     for metric, f in AMPLITUDE_FRACTIONS.items()]
    return rows, fits


dl_rows, dl_fits = [], {}
for name in CURVE_INDICES:
    for zone in ZONES:
        rows, fits = zone_dl_dates(name, zone, obs_by_zone[name][zone], base[name].level[zone])
        dl_rows += rows
        dl_fits.update(fits)
print(f"{len(dl_fits)} double-logistic fits")


def dl_curve(name, year):
    p = dl_fits.get((name, "all", year))
    return None if p is None else pd.Series(double_logistic(CURVE_DOY, *p), index=CURVE_DOY)


plot_curves("double logistic", dl_curve, dl_rows, "08_double_logistic_curves")

### 3d. All dates in one table

`season_dates` holds the dates of the candidates of `SEASON_METHODS` per zone and year (long format):

| Column | Meaning |
|---|---|
| `method`, `index`, `metric` | the candidate; metric = start, mid or end |
| `doy`, `date` | the estimated date |
| `n_obs` | clear dates of the year used for the curve |

In [ ]:
KEYS = ["method", "index", "metric"]
season_dates = pd.DataFrame(dl_rows + whittaker_rows)
is_candidate = season_dates.set_index(KEYS).index.isin([c[1:] for c in CANDIDATES])
season_dates = season_dates[is_candidate & season_dates.doy.notna()].reset_index(drop=True)

season_dates["date"] = pd.to_datetime(season_dates.year.astype(str)) + pd.to_timedelta(season_dates.doy.round() - 1, unit="D")
season_dates = season_dates.sort_values(["metric", *KEYS[:2], "zone", "year"]).reset_index(drop=True)
print(f"{len(season_dates)} dates in season_dates")

## 4. Start of the yellowing season

**Start** = the needles begin to lose chlorophyll: greenness leaves its summer level.

| Method | Start = |
|---|---|
| `Whittaker` | last day before the 50 % point with ≥ 90 % of the amplitude left (MCD12Q2 "senescence"; Gray et al. 2019) |
| `double logistic` | 90 % of the amplitude left |

There is no independent reference for the start; section 7 checks it only for its order (before the peak).

**How sections 4–6 show the dates:**
- **Table:** the whole-AOI day of year per candidate and year.
- **Figure:** the whole-AOI dates per year, one line per candidate. Where they exist, the observed dates of the two validation sources are drawn: MeteoSwiss stations in black (dashed), local sites in purple (dotted). The MeteoSwiss stations lie lower than the AOI, so compare their year-to-year ups and downs, not the level; the local sites lie at about the AOI's elevation.

In [ ]:
DATE_TYPES = list(SEASON_METHODS)
VALIDATE_WITH = {"peak": "mlard94d", "end": "mlard95d"}      # observed phase per date type
# Validation sources from 02_download_validation, kept separate: key (column prefix in section 7) ->
# (file, source, what the points are, plot style)
REFERENCES = {"ms": ("meteoswiss_larch.csv", "MeteoSwiss", "stations", dict(color="black", ls="--", marker="s")),
              "loc": ("local_larch.csv", "local", "sites", dict(color="#7b3294", ls=":", marker="D"))}
AOI_ELEV = zone_table.elev_mean["all"]   # mean elevation of the AOI pixels, shown beside the station mean

# ---- validation data per source (a source is skipped if its file is missing)
pheno = {}
for key, (file, source, _, _) in REFERENCES.items():
    if not (VALIDATION_DIR / file).exists():
        print(f"No {source} observations (run 02_download_validation)")
        continue
    p = pd.read_csv(VALIDATION_DIR / file)
    p = p[p.year.isin(years)]
    p["anomaly"] = p.doy - p.groupby(["station", "param"]).doy.transform("mean")   # each station vs its own mean
    pheno[key] = p.groupby(["param", "year"]).agg(anomaly=("anomaly", "mean"), doy=("doy", "mean"),
                                                  n_stations=("station", "nunique"), height=("height_masl", "mean"))


def reference_dates(target):
    # [(source key, label, DOY per year)] of the validation data of one date type (whole AOI)
    if target not in VALIDATE_WITH:
        return []
    refs = []
    for key, p in pheno.items():
        st = p.xs(VALIDATE_WITH[target], level="param")
        _, source, kind, _ = REFERENCES[key]
        refs.append((key, f"observed {PHENO_PARAMS[VALIDATE_WITH[target]]}, {source} {kind} (mean {st.height.mean():.0f} m)",
                     st.doy))
    return refs


def candidate_label(method, index):
    # Legend entry of a date candidate, e.g. "Whittaker curve on GRVI"
    return f"{method} curve on {index}"


def show_candidates(target, report_name=None):
    # Table and figure of the candidates of one date type (whole AOI)
    d = season_dates[season_dates.set_index(KEYS).index.isin([c[1:] for c in CANDIDATES if c[0] == target])]
    aoi = d[d.zone == "all"]
    table = aoi.pivot_table(index=KEYS, columns="year", values="doy").round(0)
    display(table)

    fig, ax = plt.subplots(figsize=(9, 4))
    for (method, index, metric), g in aoi.groupby(KEYS):
        g = g.sort_values("year")
        color, marker = METHOD_COLORS[method], INDEX_MARKERS.get(index, "o")
        ax.plot(g.year, g.doy, color=color, lw=1, marker=marker, ms=7, label=candidate_label(method, index))
    for key, label, s in reference_dates(target):
        ax.plot(s.index, s.values, lw=1.5, ms=5, label=label, **REFERENCES[key][3])
    ax.set_title(f"{date_title(target)} per year (larch area, mean {AOI_ELEV:.0f} m)")
    ax.set_xlabel("year")
    ax.set_ylabel("day of year")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
    plt.tight_layout()
    if report_name:
        save_report_figure(fig, report_name)
    plt.show()


show_candidates("start")

## 5. Peak of the yellowing

**Peak** = the canopy is most yellow: chlorophyll gone, needles still on. Both methods take the 50 % point of the autumn change, the MCD12Q2 "greendown midpoint" (Gray et al. 2019):

| Method | Peak = |
|---|---|
| `Whittaker` | first day after the summer window with ≤ 50 % of the amplitude left |
| `double logistic` | the autumn inflection day `A` |

Reference: needle colouring 50 % (`mlard94d`) at the MeteoSwiss stations and at the local sites.

In [ ]:
show_candidates("peak", "09_peak_dates_vs_observations")

## 6. End of the season

**End** = the needles have fallen: the index reaches its post-season level.

| Method | End = |
|---|---|
| `Whittaker` | first day after the 50 % point with ≤ 15 % of the amplitude left (HR-VPP end of season; Tian et al. 2021) |
| `double logistic` | 15 % of the amplitude left |

Reference: needle drop 50 % (`mlard95d`) at the MeteoSwiss stations and at the local sites. There are few clear dates after mid-October, so many end dates lie in a data gap.

In [ ]:
show_candidates("end", "10_end_dates_vs_observations")

## 7. Comparison of the two methods

One table over the candidates of sections 4–6. Complete years only.

| Column | Meaning |
|---|---|
| `coverage` | share of zone-years with a date |
| `order_ok` | share of zone-years in the order start < peak < end, judged against the median of the candidates of the neighbouring date types |
| `days_per_100m` | within-year elevation gradient across zones; negative = earlier higher up, as expected (elevation controls larch phenology; Migliavacca et al. 2008) |
| `aoi_sd_years` | year-to-year standard deviation of the whole-AOI date |
| `ms_r`, `ms_rmse`, `ms_n` | whole-AOI anomalies vs MeteoSwiss station anomalies: needle colouring 50 % for the peak, needle drop 50 % for the end. Correlation, RMSE in days, number of years |
| `ms_offset` | mean AOI date − mean station date. The stations lie lower (1050–1930 m), so this is not an error |
| `loc_r`, `loc_rmse`, `loc_n`, `loc_offset` | the same against the local sites (`local_larch.csv`). They lie at 1720–2090 m, about the AOI's elevation, so `loc_offset` also checks the level |

**Figure:** the whole-AOI mean date ± sd over the years of every candidate; vertical lines = mean date of the reference data (black dashed: MeteoSwiss, purple dotted: local sites). With about 9 years, a correlation needs |r| > 0.67 to be significant at 5 %.

In [ ]:
d = season_dates.dropna(subset=["doy"])
n_zone_years = (len(ZONES) - 1) * len(years)
# ---- candidates with dates and the consensus (median of the candidates) per zone-year and date type
candidates = {}
for c in CANDIDATES:
    g = d[(d.method == c[1]) & (d["index"] == c[2]) & (d.metric == c[3])]
    if not g.empty:
        candidates[c] = g.set_index(["zone", "year"]).doy
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # zone-years without any candidate of a date type
    consensus = pd.DataFrame({t: pd.concat([s for k, s in candidates.items() if k[0] == t], axis=1).median(axis=1)
                              for t in DATE_TYPES})


def order_share(target, s):
    c = consensus.reindex(s.index)
    ok = {"start": s < c["peak"], "peak": (c["start"] < s) & (s < c["end"]), "end": s > c["peak"]}[target]
    known = {"start": c["peak"].notna(), "peak": c["start"].notna() & c["end"].notna(), "end": c["peak"].notna()}[target]
    return ok[known].mean() if known.any() else np.nan


def elevation_slope(s):
    # Days per 100 m from zone dates, within years (each year's mean removed from date and elevation)
    z = s.drop(index="all", level="zone", errors="ignore").reset_index()
    z["x"] = z.zone.map(zone_table.elev_mean) / 100
    z = z[z.groupby("year").x.transform("size") >= 2]
    x, y = z.x - z.groupby("year").x.transform("mean"), z.doy - z.groupby("year").doy.transform("mean")
    return (x @ y) / (x @ x) if len(z) > 3 and (x @ x) > 0 else np.nan


def reference_check(target, aoi):
    # Whole-AOI anomalies vs the anomalies of each validation source: columns <key>_r, _rmse, _n, _offset
    out = {}
    if target not in VALIDATE_WITH:
        return out
    for key, p in pheno.items():
        st = p.xs(VALIDATE_WITH[target], level="param")
        common = aoi.index.intersection(st.index)
        if len(common) < 4:
            out[f"{key}_n"] = len(common)
            continue
        a = aoi[common] - aoi[common].mean()
        s = st.anomaly[common] - st.anomaly[common].mean()
        out.update({f"{key}_r": a.corr(s), f"{key}_rmse": np.sqrt(((a - s) ** 2).mean()), f"{key}_n": len(common),
                    f"{key}_offset": (aoi[common] - st.doy[common]).mean()})
    return out


rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # medians / means of candidates without any value
    for (target, method, index, metric), s in candidates.items():
        g = d[(d.method == method) & (d["index"] == index) & (d.metric == metric)]
        gz = g[g.zone != "all"]
        aoi = s.xs("all", level="zone") if "all" in s.index.get_level_values("zone") else pd.Series(dtype=float)
        rows.append({"target": target, "method": method, "index": index, "metric": metric,
                     "coverage": len(gz) / n_zone_years,
                     "order_ok": order_share(target, s.drop(index="all", level="zone", errors="ignore")),
                     "days_per_100m": elevation_slope(s), "aoi_mean_doy": aoi.mean(), "aoi_sd_years": aoi.std(),
                     **reference_check(target, aoi)})
comparison = pd.DataFrame(rows).set_index(["target", *KEYS])
display(comparison.round(2))

# ---- figure: mean date +- sd over the years per candidate (whole AOI)
fig, axes = plt.subplots(1, len(DATE_TYPES), figsize=(5.5 * len(DATE_TYPES), 3.2), squeeze=False)
for ax, target in zip(axes[0], DATE_TYPES):
    t = comparison.xs(target, level="target")
    for k, ((method, index, metric), r) in enumerate(t.iterrows()):
        ax.errorbar(r.aoi_mean_doy, k, xerr=r.aoi_sd_years, fmt=INDEX_MARKERS.get(index, "o"),
                    color=METHOD_COLORS[method], ms=7, capsize=3)
    for key, label, s in reference_dates(target):
        style = REFERENCES[key][3]
        ax.axvline(s.mean(), color=style["color"], ls=style["ls"], lw=1.2, label=label)
    ax.set_yticks(range(len(t)), [" / ".join(key) for key in t.index], fontsize=8)
    ax.set_ylim(-0.6, len(t) - 0.4)
    ax.set_title(date_title(target))
    ax.set_xlabel("day of year")
    ax.grid(alpha=0.3)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=7, loc="lower left")
fig.suptitle(f"Mean date ± sd over the years per candidate (whole AOI, mean {AOI_ELEV:.0f} m)")
plt.tight_layout()
plt.show()

### 7b. Chosen dates

`SEASON_CHOICE` in `config.py` sets the date used in sections 8 and 9, per date type:
- `"average"`: the mean of both candidates. Each enters as its anomaly from its own mean over all zone-years, plus the mean level of both, so a candidate missing in one year does not shift that year's level.
- one of the two candidates, written as in `SEASON_METHODS`, e.g. `"Whittaker / GRVI / mid"`.

`selected` holds the chosen start / peak / end per zone and year (complete years). The figure shows the season of the whole AOI per year with the reference data.

In [ ]:
def chosen_keys(target):
    keys = [k[1:] for k in candidates if k[0] == target]
    choice = SEASON_CHOICE[target]
    if choice == "average":
        if not keys:
            raise ValueError(f"No {target} candidate has dates; check SEASON_METHODS[{target!r}]")
        return keys
    key = tuple(part.strip() for part in choice.split("/"))
    if key not in keys:
        options = [" / ".join(k) for k in keys]
        raise ValueError(f"SEASON_CHOICE[{target!r}] = {choice!r} is not a {target} candidate with dates. Options: {options}")
    return [key]


def combine(target, keys):
    # Mean of the chosen candidates per zone-year: each candidate's anomaly from its own mean over all zone-years,
    # averaged over the candidates present, plus the mean level of all of them
    s = pd.concat([candidates[(target, *k)] for k in keys], axis=1, ignore_index=True)
    return s.sub(s.mean()).mean(axis=1) + s.mean().mean()


def doy_to_date(year, doy):
    return date(int(year), 1, 1) + timedelta(days=int(round(doy)) - 1) if pd.notna(doy) else None


chosen = {target: chosen_keys(target) for target in DATE_TYPES}
choice_label = {t: f"average of {len(k)}" if len(k) > 1 else " / ".join(k[0]) for t, k in chosen.items()}
# The same in words, for figure titles and legends
choice_text = {t: "average of the " + " and the ".join(candidate_label(*k[:2]) for k in keys) if len(keys) > 1
               else candidate_label(*keys[0][:2]) for t, keys in chosen.items()}
for target, keys in chosen.items():
    print(f"{target}: {choice_label[target]}: {'; '.join(' / '.join(k) for k in keys)}")
selected = pd.DataFrame({target: combine(target, keys) for target, keys in chosen.items()}).sort_index()

z = selected.drop(index="all", level="zone").dropna()
print(f"start < peak < end in {((z.start < z.peak) & (z.peak < z.end)).mean():.0%} of {len(z)} zone-years")
aoi_sel = selected.xs("all", level="zone")
display(aoi_sel.round(0).join(pd.DataFrame({f"{t}_date": [doy_to_date(y, v) for y, v in aoi_sel[t].items()]
                                            for t in DATE_TYPES}, index=aoi_sel.index)))

MONTH_TICKS = {date(2023, m, 1).timetuple().tm_yday: f"1 {date(2023, m, 1):%b}" for m in range(8, 13)}
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.vlines(aoi_sel.index, aoi_sel.start, aoi_sel.end, color="tab:olive", lw=10, alpha=0.35, label="start to end")
ax.plot(aoi_sel.index, aoi_sel.start, "_", color="tab:olive", ms=18, mew=2, label="start")
ax.plot(aoi_sel.index, aoi_sel.peak, "o", color="darkorange", ms=9, label="peak")
ax.plot(aoi_sel.index, aoi_sel.end, "_", color="saddlebrown", ms=18, mew=2, label="end")
for target, marker in [("peak", "s"), ("end", "D")]:
    for key, label, s in reference_dates(target):
        ax.plot(s.index + (0.2 if key == "ms" else -0.2), s.values, marker, color=REFERENCES[key][3]["color"],
                mfc="white", ms=6, ls="", label=label)
ax.set_yticks(list(MONTH_TICKS), [f"{v} ({k})" for k, v in MONTH_TICKS.items()])
ax.set_ylabel("date (day of year)")
ax.set_xlabel("year")
ax.set_xticks(aoi_sel.index)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
ax.set_title(f"Start, peak and end of the yellowing per year (larch area, mean {AOI_ELEV:.0f} m)")
plt.tight_layout()
save_report_figure(fig, '11_season_dates_chosen')
plt.show()

### 7c. Check at a local site

Section 7 compares whole-AOI dates with the mean of stations or sites elsewhere. Here the dates are computed for the AOI pixels within `SITE_RADIUS_M` of each site in `SITE_AREAS` (statistics from `01` section 6) and compared with that site's own observations in `local_larch.csv`, year by year.

- **Dates:** computed as for the zones: date filter (section 1; the scene must cover `MIN_COVERED_PIXELS` of the area, or all of it if it is smaller), weights and outliers (3a), λ of 3b, both methods and the candidates of `SEASON_METHODS`. Post-season level: the area's own, or the whole AOI's with fewer than `MIN_BASE_OBS` dates (3a).
- **Comparison** (`site_check`), per site and candidate, over the complete years with an estimate and an observation, as in section 7: `r` and `rmse` of the anomalies (each series minus its own mean), `n` years, `offset` = mean estimated − observed date. Area and site are in the same place, so `offset` is a bias here, apart from the elevation difference between the area (`elev_mean`) and the site. `mae` = mean absolute difference of the dates themselves.
- With 8 years, `r` needs > 0.71 to be significant at 5 %.
- **Figures:** per site, the estimated dates of both candidates (colours as in sections 4–6) and the observed dates (purple), first for the peak and the end, then for the peak alone.

In [ ]:
site_dates, site_check = pd.DataFrame(), pd.DataFrame()
observed_file = VALIDATION_DIR / "local_larch.csv"
if not SITE_AREAS:
    print("No SITE_AREAS in config.py")
elif not (SITE_CACHE_DIR / "zones.csv").exists() or not observed_file.exists():
    print("No site statistics or local observations (run 01_download_sentinel2 section 6 and 02_download_validation)")
else:
    site_stats = load_stats(SITE_CACHE_DIR)
    site_zones = load_zone_table(SITE_CACHE_DIR)
    site_years = [y for y in years if y not in incomplete_years(site_stats)]
    observed = pd.read_csv(observed_file)
    observed = observed[observed.year.isin(site_years)]
    site_names = pd.read_csv(VALIDATION_DIR / "local_sites.csv", index_col="station").name

    # ---- dates of the site areas, computed as for the zones (3a-3c)
    rows = []
    for site in site_zones.index:
        for name in CURVE_INDICES:
            obs = obs_frame(name, site, site_stats, site_zones)
            post = obs[obs.doy.between(*POST_WINDOW) & (obs.w > 0)]
            lo = post.y.median() if len(post) >= MIN_BASE_OBS else base[name].level["all"]
            obs = flag_outliers(obs, lo)
            rows += zone_whittaker_dates(name, site, obs, lo, smooth_series(obs, LAMBDA[name], lo))
            rows += zone_dl_dates(name, site, obs, lo)[0]
    site_dates = pd.DataFrame(rows).rename(columns={"zone": "site"})
    site_dates = site_dates[site_dates.set_index(KEYS).index.isin([c[1:] for c in CANDIDATES])
                            & site_dates.doy.notna() & site_dates.year.isin(site_years)].reset_index(drop=True)

    # ---- estimated vs observed dates per site and candidate
    def site_series(site, candidate):
        g = site_dates[(site_dates.site == site) & (site_dates.method == candidate[1])
                       & (site_dates["index"] == candidate[2]) & (site_dates.metric == candidate[3])]
        return g.set_index("year").doy.sort_index()

    check_rows = []
    for site in site_zones.index:
        for target, param in VALIDATE_WITH.items():
            seen = observed[(observed.station == site) & (observed.param == param)].set_index("year").doy
            for c in [c for c in CANDIDATES if c[0] == target]:
                est = site_series(site, c)
                common = est.index.intersection(seen.index)
                row = {"site": site, "target": target, "method": c[1], "index": c[2], "metric": c[3], "n": len(common)}
                if len(common) >= 4:
                    a, s = est[common] - est[common].mean(), seen[common] - seen[common].mean()
                    row.update(r=a.corr(s), rmse=np.sqrt(((a - s) ** 2).mean()),
                               offset=(est[common] - seen[common]).mean(), mae=(est[common] - seen[common]).abs().mean())
                check_rows.append(row)
    site_check = pd.DataFrame(check_rows).set_index(["site", "target", *KEYS])
    display(site_zones.join(site_names).round(1))
    display(site_check.round(2))

    # ---- figures per site: estimated and observed dates per year (peak and end; then the peak alone)
    def plot_site(site, targets, report_name=None):
        fig, axes = plt.subplots(1, len(targets), figsize=(6 * len(targets), 4.2), squeeze=False)
        for ax, target in zip(axes[0], targets):
            param = VALIDATE_WITH[target]
            for c in [c for c in CANDIDATES if c[0] == target]:
                est = site_series(site, c)
                ax.plot(est.index, est.values, color=METHOD_COLORS[c[1]], marker=INDEX_MARKERS.get(c[2], "o"), lw=1,
                        ms=6, label=candidate_label(c[1], c[2]))
            seen = observed[(observed.station == site) & (observed.param == param)].sort_values("year")
            ax.plot(seen.year, seen.doy, lw=1.5, ms=5, label="observed at the site", **REFERENCES["loc"][3])
            ax.set_title(f"{date_title(target)}\nvs observed {PHENO_PARAMS[param]}")
            ax.set_xlabel("year")
            ax.xaxis.get_major_locator().set_params(integer=True)
            ax.grid(alpha=0.3)
        axes[0, 0].set_ylabel("day of year")
        z = site_zones.loc[site]
        fig.suptitle(f"{site_names.get(site, site)}: larch within {SITE_RADIUS_M} m of the site vs the site's observations\n"
                     f"({z.n_pixels:.0f} pixels, mean {z.elev_mean:.0f} m)")
        fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="lower center", ncol=3, fontsize=9)
        plt.tight_layout(rect=(0, 0.08, 1, 1))
        if report_name:
            save_report_figure(fig, report_name)
        plt.show()

    for site in site_zones.index:
        plot_site(site, list(VALIDATE_WITH), f"12_site_check_{site}")
        plot_site(site, ["peak"])

## 8. Trend and short-term projection (no climate drivers)

For each chosen date (7b) of the whole AOI and the complete years:
- **OLS trend** of the day of year on the year, with its 95 % confidence interval, extrapolated `FORECAST_YEARS` ahead with a 95 % prediction interval (the range a single future year is expected to fall in).
- **Mean model** (no trend) alongside. If the trend is not significant (`trend_significant` = False), the mean ± interval is the more honest expectation.

With about 9 years, one early or late year at either end moves the slope a lot. This is the statistical baseline that the climate models in `05` are compared against.

In [ ]:
proj_rows = []
level = f"{1 - ALPHA:.0%}"
fig, axes = plt.subplots(1, len(DATE_TYPES), figsize=(5.5 * len(DATE_TYPES), 4.5), squeeze=False)
for ax, target in zip(axes[0], DATE_TYPES):
    d = aoi_sel[target].dropna()
    n = len(d)
    if n < 4:
        ax.set_title(f"{target}: only {n} years, skipped")
        continue
    years_fit = d.index.values.astype(float)
    y = d.values.astype(float)
    x0 = years_fit.mean()   # centred years for numerical stability
    ols = sm.OLS(y, sm.add_constant(years_fit - x0)).fit()
    mean_model = sm.OLS(y, np.ones((n, 1))).fit()
    future = np.arange(years_fit.max() + 1, years_fit.max() + 1 + FORECAST_YEARS)
    grid = np.arange(years_fit.min(), future.max() + 1)
    pred = ols.get_prediction(sm.add_constant(grid - x0, has_constant="add")).summary_frame(alpha=ALPHA)
    pred_mean = mean_model.get_prediction(np.ones((len(grid), 1))).summary_frame(alpha=ALPHA)
    slope_lo, slope_hi = ols.conf_int(ALPHA)[1]
    significant = bool(ols.pvalues[1] < ALPHA)
    print(f"{target} ({choice_label[target]}), {n} years: OLS slope {ols.params[1]:+.2f} d/yr "
          f"({level} CI {slope_lo:+.2f} to {slope_hi:+.2f}), p = {ols.pvalues[1]:.3f}, R2 = {ols.rsquared:.2f}")
    for i, yr in enumerate(grid):
        if yr in future:
            proj_rows.append({
                "date_type": target, "year": int(yr),
                "trend_doy": pred["mean"].iloc[i],
                "trend_pi_low": pred["obs_ci_lower"].iloc[i], "trend_pi_high": pred["obs_ci_upper"].iloc[i],
                "mean_doy": pred_mean["mean"].iloc[i],
                "mean_pi_low": pred_mean["obs_ci_lower"].iloc[i], "mean_pi_high": pred_mean["obs_ci_upper"].iloc[i],
                "trend_significant": significant,
            })

    color = "tab:blue"
    ax.fill_between(grid, pred["obs_ci_lower"], pred["obs_ci_upper"], color=color, alpha=0.15,
                    label=f"{level} range of a single year (trend)")
    ax.fill_between(grid, pred["mean_ci_lower"], pred["mean_ci_upper"], color=color, alpha=0.3,
                    label=f"{level} confidence band of the trend")
    ax.plot(grid, pred["mean"], color=color, lw=2, label="linear trend")
    ax.plot(grid, pred_mean["mean"], "--", color="grey", lw=1, label="mean of all years")
    ax.fill_between(grid, pred_mean["obs_ci_lower"], pred_mean["obs_ci_upper"], facecolor="none",
                    edgecolor="grey", hatch="//", lw=0, alpha=0.5, label=f"{level} range of a single year (mean)")
    ax.plot(years_fit, y, "o", color="black", label="estimated date")
    ax.axvspan(future.min() - 0.5, future.max() + 0.5, color="grey", alpha=0.08)
    ax.set_title(f"{date_title(target)}\ntrend {ols.params[1]:+.2f} days per year (p = {ols.pvalues[1]:.2f})", fontsize=10)
    ax.set_xlabel("year")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("day of year")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(handles), fontsize=8)
plt.suptitle("Trend of the dates and short projection (whole larch area; shaded = projected years)")
plt.tight_layout(rect=(0, 0.07, 1, 1))
save_report_figure(fig, '13_trend_projection')
plt.show()

season_projection = pd.DataFrame(proj_rows)
if not season_projection.empty:
    season_projection = season_projection.set_index(["date_type", "year"]).round(1)
    for c in ["trend_doy", "mean_doy"]:
        season_projection[c.replace("_doy", "_date")] = [doy_to_date(yr, v) for (_, yr), v in season_projection[c].items()]
    display(season_projection)

## 9. Phenology vs topography (elevation band × aspect)

Uses the chosen dates of 7b per zone and year.

- **Map:** elevation band and aspect class of the AOI pixels (`zones.tif` from `01`) on a hillshade. The hillshade is lit from the south, so north-facing slopes appear dark: blue (N) pixels should sit on the dark side of the valleys.
- **Table:** zone attributes, the mean number of clear autumn dates per year and index (`autumn_dates_*`, `AUTUMN_START_DOY`–`POST_WINDOW[1]`) and the mean share of SCL "dark" pixels in autumn.
- **Plots:** day of year against elevation band, one line per aspect (mean ± sd over years), first for all date types, then for the peak alone with the local sites only. The validation stations and sites (section 7: MeteoSwiss black squares, local sites purple diamonds) are drawn at their own height with their mean ± sd over the complete years: needle colouring 50 % for the peak, needle drop 50 % for the end.
- **Model:** OLS `doy ~ elev_100m + C(aspect) + C(year)`.
  - The elevation coefficient is in days per 100 m.
  - The aspect coefficient is the north − south difference in days.
  - `C(year)` absorbs year-to-year weather.

**Caveats:**
- Zones share weather and clouds within a year, so the CIs are somewhat optimistic.
- North slopes lose more autumn dates to SCL class 2 (dark / topographic shadow); see `autumn_dates_*` and `autumn_dark_fraction`.

In [ ]:
zr = rioxarray.open_rasterio(CACHE_DIR / "zones.tif", masked=True)
elev, elev_band, aspect = (zr.sel(band=b).values for b in (1, 2, 3))
on = ~np.isnan(elev_band)
yy, xx = np.meshgrid(zr.y.values, zr.x.values, indexing="ij")
left, bottom, right, top = zr.rio.bounds()
shade = LightSource(azdeg=180, altdeg=30).hillshade(np.nan_to_num(elev, nan=np.nanmean(elev)),
                                                   dx=RESOLUTION, dy=RESOLUTION)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax in axes:
    ax.imshow(shade, cmap="gray", extent=(left, right, bottom, top), alpha=0.6)
bands = np.unique(elev_band[on])
for b, color in zip(bands, plt.get_cmap("viridis")(np.linspace(0, 1, len(bands)))):
    sel = on & (elev_band == b)
    axes[0].scatter(xx[sel], yy[sel], s=0.2, color=color, label=f"{int(b):,}–{int(b + ELEV_STEP):,} m")
for asp, code in ASPECT_CODES.items():
    sel = on & (aspect == code)
    axes[1].scatter(xx[sel], yy[sel], s=0.2, color=ASPECT_COLORS[asp], label=f"{ASPECT_NAMES[asp]} ({int(sel.sum()):,} pixels)")
for ax, title in zip(axes, ["Elevation of the larch pixels", "Aspect of the larch pixels (hillshade lit from the south)"]):
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.legend(markerscale=20, fontsize=8, loc="lower right")
    ax.ticklabel_format(style="plain", useOffset=False)
    ax.set_xlabel("easting (m)")
axes[0].set_ylabel("northing (m)")
plt.tight_layout()
save_report_figure(fig, '05_topographic_zones')
plt.show()

In [ ]:
zone_attrs = zone_table.drop(index="all")[["elev_band", "aspect", "elev_mean"]]
autumn = stats_all[stats_all.index.dayofyear >= AUTUMN_START_DOY]
autumn_dates = pd.DataFrame({
    f"autumn_dates_{name}": {zone: o[(o.w > 0) & o.doy.between(AUTUMN_START_DOY, POST_WINDOW[1])]
                             .groupby("year").size().mean() for zone, o in obs_by_zone[name].items()}
    for name in INDICES})
coverage = (zone_table[["elev_band", "aspect", "n_pixels", "elev_mean", "slope_mean", "northness_mean"]]
            .join(autumn_dates)
            .join(autumn.groupby("zone").scl_dark_fraction.mean().rename("autumn_dark_fraction")))
display(coverage.round(2))

zone_dates = selected.drop(index="all", level="zone").reset_index().join(zone_attrs, on="zone", how="inner")

def station_dates(target, sources=REFERENCES):
    # Validation data of one date type per station / site: [(source key, name, height, mean and sd of the DOY over
    # the complete years)]; sources: keys of REFERENCES
    if target not in VALIDATE_WITH:
        return []
    out = []
    for key, (file, _, _, _) in REFERENCES.items():
        if key not in sources or not (VALIDATION_DIR / file).exists():
            continue
        p = pd.read_csv(VALIDATION_DIR / file)
        p = p[(p.param == VALIDATE_WITH[target]) & p.year.isin(years)]
        out.append((key, p.groupby("station").agg(name=("name", "first"), height=("height_masl", "first"),
                                                  mean=("doy", "mean"), std=("doy", "std"))))
    return out


def plot_topography(targets, names=False, sources=REFERENCES, title=None, zone_names=None, report_name=None):
    # Chosen dates per elevation band and aspect, with the validation stations / sites at their own height;
    # names: label each station / site (above its whisker); sources: keys of REFERENCES to draw;
    # title: replaces the figure and panel titles; zone_names: legend entry of the zone lines per aspect
    width = 12 if len(targets) == 1 else 5.5 * len(targets) + 1
    fig, axes = plt.subplots(1, len(targets), figsize=(width, 5.5), sharex=True, squeeze=False, layout="constrained")
    for ax, target in zip(axes[0], targets):
        for asp, color in ASPECT_COLORS.items():
            d = zone_dates[zone_dates.aspect == asp].dropna(subset=[target])
            if d.empty:
                continue
            shift = ELEV_STEP / 2 + {"N": -0.08, "S": 0.08, "flat": 0}[asp] * ELEV_STEP   # band centre, dodged
            ax.scatter(d.elev_band + shift, d[target], s=8, color=color, alpha=0.3)
            g = d.groupby("elev_band")[target].agg(["mean", "std"])
            ax.errorbar(g.index + shift, g["mean"], yerr=g["std"].fillna(0), fmt="o-", color=color, capsize=3,
                        label=(zone_names or {}).get(asp, f"larch zones, {ASPECT_NAMES[asp]}"))
        tags = []
        for key, st in station_dates(target, sources):
            _, source, kind, style = REFERENCES[key]
            ax.errorbar(st.height, st["mean"], yerr=st["std"].fillna(0), fmt=style["marker"], color=style["color"],
                        mfc="white", ms=7, capsize=3, label=f"observed {PHENO_PARAMS[VALIDATE_WITH[target]]}, {source} {kind}")
            tags += [(r.height, r["mean"] + np.nan_to_num(r["std"]), r["name"], style["color"]) for _, r in st.iterrows()]
        if names:
            last, dx = -np.inf, 0
            for height, y, name, color in sorted(tags):
                dx = dx + 9 if height - last < 25 else 0   # names of stations closer than 25 m side by side
                last = height
                ax.annotate(name, (height, y), xytext=(dx, 4), textcoords="offset points", rotation=90, ha="center",
                            va="bottom", fontsize=7, color=color)
            lo, hi = ax.get_ylim()
            ax.set_ylim(lo, hi + 0.2 * (hi - lo))   # room for the names
        if title is None:
            ax.set_title(date_title(target), fontsize=10)
        ax.set_xlabel("elevation (m)")
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel("day of year")
    labels = {}
    for ax in axes[0]:
        for h, label in zip(*ax.get_legend_handles_labels()):
            labels.setdefault(label, h)
    fig.legend(labels.values(), labels.keys(), loc="outside lower center", ncol=3, fontsize=8)
    fig.suptitle(title or "Dates by elevation and aspect (mean ± sd over years)", fontsize=12)
    if report_name:
        save_report_figure(fig, report_name)
    plt.show()


plot_topography(DATE_TYPES, report_name="14_dates_vs_elevation")
plot_topography(["peak"], names=True, sources=["loc"], title="Peak of the yellowing by elevation and aspect (mean ± sd over years)",
                report_name="15_peak_vs_elevation")

NS_TERM = "C(aspect, Treatment('S'))[T.N]"


def topo_model(data, col):
    d = data.dropna(subset=[col]).assign(elev_100m=lambda x: x.elev_mean / 100)
    if (not {"N", "S"} <= set(d.aspect) or d.elev_band.nunique() < 2
            or len(d) < d.year.nunique() + d.aspect.nunique() + 3):
        return None
    return smf.ols(f"{col} ~ elev_100m + C(aspect, Treatment('S')) + C(year)", data=d).fit()
rows = []
for target in DATE_TYPES:
    fit = topo_model(zone_dates, target)
    if fit is None:
        print(f"{target}: too few zone-years (needs N and S zones and >= 2 elevation bands), no model")
        continue
    ci = fit.conf_int(ALPHA)
    rows.append({
        "date_type": target, "method": choice_label[target], "n_zone_years": int(fit.nobs),
        "n_zones": zone_dates.dropna(subset=[target]).zone.nunique(),
        "days_per_100m": fit.params["elev_100m"],
        "elev_ci_low": ci.loc["elev_100m", 0], "elev_ci_high": ci.loc["elev_100m", 1],
        "elev_p": fit.pvalues["elev_100m"],
        "north_minus_south_days": fit.params[NS_TERM],
        "ns_ci_low": ci.loc[NS_TERM, 0], "ns_ci_high": ci.loc[NS_TERM, 1], "ns_p": fit.pvalues[NS_TERM],
        "r2": fit.rsquared,
    })
topo_effects = pd.DataFrame(rows).set_index("date_type").round(3) if rows else pd.DataFrame()
display(topo_effects)

## 10. Trend / seasonal / remainder decomposition (STL, whole AOI)

Robust STL on a regular 5-day grid splits each index into a long-term trend, a repeating yearly cycle and the remainder. It needs at least two full years.

In [ ]:
STEP_DAYS = 5
PERIOD = int(round(365.25 / STEP_DAYS))   # observations per year on the regular grid


def stl_decompose(series, name):
    s = series.resample(f"{STEP_DAYS}D").mean()
    s = s.loc[s.first_valid_index():s.last_valid_index()].interpolate(method="time")
    if len(s) < 2 * PERIOD:
        print(f"{name}: {len(s) * STEP_DAYS / 365.25:.1f} years of data; STL needs at least 2 full years")
        return None
    res = STL(s, period=PERIOD, robust=True).fit()
    return pd.DataFrame({"observed": s, "trend": res.trend, "seasonal": res.seasonal, "remainder": res.resid})


for name in INDICES:
    dec = stl_decompose(medians[name], name)
    if dec is None:
        continue
    color = INDEX_COLORS[name]
    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
    axes[0].plot(dec.index, dec.observed, color="grey", lw=1)
    axes[0].set_ylabel(f"{name} observed")
    axes[1].plot(dec.index, dec.trend, color=color, lw=2)
    axes[1].set_ylabel("trend")
    axes[2].plot(dec.index, dec.seasonal, color=color, lw=1.5)
    axes[2].set_ylabel("seasonal")
    axes[3].scatter(dec.index, dec.remainder, s=6, color=color)
    axes[3].set_ylabel("remainder")
    for a in axes[2:]:
        a.axhline(0, color="black", lw=0.5)
    for a in axes:
        a.grid(alpha=0.3)
    axes[0].set_title(f"{name} STL decomposition (period = {PERIOD} x {STEP_DAYS} days)")
    plt.tight_layout()
    plt.show()
    slope = np.polyfit((dec.index - dec.index[0]).days / 365.25, dec.trend.values, 1)[0]
    print(f"{name} trend slope: {slope:+.4f} per year | seasonal amplitude: "
          f"{dec.seasonal.max() - dec.seasonal.min():.3f} | remainder std: {dec.remainder.std():.3f}")

## 11. Export

The figures recommended for the report are saved to `report_figures/` while the notebook runs (PNG, 300 dpi, numbered in report order). The maps of `02_download_validation` are added here as JPG, as they hold aerial and satellite images; `05_climate_analysis` adds its own figures.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_DIR / "ndvi_timeseries_all_dates.csv")
df_clean.to_csv(OUT_DIR / "ndvi_timeseries_clean.csv")
season_dates.to_csv(OUT_DIR / "season_dates.csv", index=False)
comparison.to_csv(OUT_DIR / "season_method_comparison.csv")
selected.to_csv(OUT_DIR / "season_selected.csv")
season_projection.to_csv(OUT_DIR / "season_projection.csv")
zone_table.to_csv(OUT_DIR / "zones.csv")
topo_effects.to_csv(OUT_DIR / "topography_effects.csv")
if not site_dates.empty:
    site_dates.to_csv(OUT_DIR / "site_dates.csv", index=False)
    site_check.to_csv(OUT_DIR / "site_check.csv")
# Maps of 02_download_validation recommended for the report
REPORT_MAPS = {"station_map.png": "01_station_map", "aoi_map.png": "02_aoi_national_park",
               "aoi_reference_map.png": "03_aoi_vs_forest_type_map", "aoi_imagery_map.png": "04_aoi_over_images"}
for file, name in REPORT_MAPS.items():
    if (VALIDATION_DIR / file).exists():
        Image.open(VALIDATION_DIR / file).convert("RGB").save(REPORT_FIG_DIR / f"{name}.jpg", quality=90)
    else:
        print(f"{file} missing (run 02_download_validation)")
print(f"Results written to {OUT_DIR}; figures for the report in {REPORT_FIG_DIR}")

## References

- Atzberger, C. & Eilers, P. H. C. (2011). A time series for monitoring vegetation activity and phenology at 10-daily time steps covering large parts of South America. *International Journal of Digital Earth* 4(5), 365–386.
- Beck, P. S. A., Atzberger, C., Høgda, K. A., Johansen, B. & Skidmore, A. K. (2006). Improved monitoring of vegetation dynamics at very high latitudes: a new method using MODIS NDVI. *Remote Sensing of Environment* 100(3), 321–334.
- Busetto, L., Colombo, R., Migliavacca, M., et al. (2010). Remote sensing of larch phenological cycle and analysis of relationships with climate in the Alpine region. *Global Change Biology* 16(9), 2504–2517.
- Copernicus Land Monitoring Service (2021). *HR-VPP Algorithm Theoretical Basis Document: Seasonal Trajectories and VPP parameters*, issue 1.1. https://land.copernicus.eu/en/technical-library
- Eilers, P. H. C. (2003). A perfect smoother. *Analytical Chemistry* 75(14), 3631–3636.
- Gray, J., Sulla-Menashe, D. & Friedl, M. A. (2019). *User Guide to Collection 6 MODIS Land Cover Dynamics (MCD12Q2) Product*. NASA LP DAAC.
- Jönsson, P. & Eklundh, L. (2004). TIMESAT – a program for analyzing time-series of satellite sensor data. *Computers & Geosciences* 30(8), 833–845.
- MeteoSwiss. Phenological observations, open data (`ch.meteoschweiz.ogd-phenology`). https://opendatadocs.meteoswiss.ch/a-data-groundbased/a9-phenological-observations
- Local larch phenology observations (`Lärchen_pheno.xlsx`): sites Il Fuorn, Val Cluozza, Val Mingèr, Val Trupchun and Lü.
- Merzlyak, M. N., Gitelson, A. A., Chivkunova, O. B. & Rakitin, V. Y. (1999). Non-destructive optical detection of pigment changes during leaf senescence and fruit ripening. *Physiologia Plantarum* 106(1), 135–141.
- Migliavacca, M., Cremonese, E., Colombo, R., et al. (2008). European larch phenology in the Alps: can we grasp the role of ecological factors by combining field observations and inverse modelling? *International Journal of Biometeorology* 52(7), 587–605.
- Motohka, T., Nasahara, K. N., Oguma, H. & Tsuchida, S. (2010). Applicability of green-red vegetation index for remote sensing of vegetation phenology. *Remote Sensing* 2(10), 2369–2387.
- Tian, F., Cai, Z., Jin, H., et al. (2021). Calibrating vegetation phenology from Sentinel-2 using eddy covariance, PhenoCam, and PEP725 networks across Europe. *Remote Sensing of Environment* 260, 112456.
- Zeng, L., Wardlow, B. D., Xiang, D., Hu, S. & Li, D. (2020). A review of vegetation phenological metrics extraction using time-series, multispectral satellite data. *Remote Sensing of Environment* 237, 111511.